---

## 종합 예제: 사용자 컨텍스트 기반 에이전트

Runtime의 모든 기능을 활용한 실용적인 예제입니다. Context로 사용자 정보를 전달하고, Store로 검색 기록을 관리하며, 미들웨어로 동적 프롬프트와 사용량 추적을 구현합니다.

아래 코드는 사용자 등급에 따라 다른 기능을 제공하는 에이전트 예시입니다.

In [1]:
from feature.RuntimeBasic import RuntimeBasicAgent, UserContext
from langchain_core.messages import HumanMessage

workflow = RuntimeBasicAgent()

In [2]:
# 테스트 1: 엔터프라이즈 사용자 (한국어)
print("=== Test 1: Enterprise User (Korean) ===")
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Search for 'machine learning'"}]},
    context=UserContext(
        user_id="user_001", user_name="김철수", user_tier="enterprise", language="ko"
    ),
)

=== Test 1: Enterprise User (Korean) ===
[Usage Tracker] User: user_001, Tier: enterprise

🔄 Node: track_usage.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  search_database (call_uNuOXIhJAnddSmZKfl7zzywB)
 Call ID: call_uNuOXIhJAnddSmZKfl7zzywB
  Args:
    query: machine learning

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: search_database

Full database search results for: machine learning (Enterprise access)
[Usage Tracker] User: user_001, Tier: enterprise

🔄 Node: track_usage.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Cal

In [3]:
print("\n=== Test 2: Free User (English) ===")
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Search for 'data science'"}]},
    context=UserContext(
        user_id="user_002", user_name="John Doe", user_tier="free", language="en"
    ),
)


=== Test 2: Free User (English) ===
[Usage Tracker] User: user_002, Tier: free

🔄 Node: track_usage.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  search_database (call_f2Q7f8bUbolIhBldbB8wPlMK)
 Call ID: call_f2Q7f8bUbolIhBldbB8wPlMK
  Args:
    query: data science

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: search_database

Basic search results for: data science (Limited to 10 results)
[Usage Tracker] User: user_002, Tier: free

🔄 Node: track_usage.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  save_search (call_Krbf

In [6]:
# 테스트 3: 검색 기록 조회
print("\n=== Test 3: Check Search History ===")
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "What's my search history?"}]},
    context=UserContext(
        user_id="user_001", user_name="김철수", user_tier="enterprise", language="ko"
    ),
)


=== Test 3: Check Search History ===
[Usage Tracker] User: user_001, Tier: enterprise

🔄 Node: track_usage.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_user_history (call_UbEFxlSnjVJj0pb1F1oa9m1f)
 Call ID: call_UbEFxlSnjVJj0pb1F1oa9m1f
  Args:

🔄 Node: tools 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_user_history

Recent searches: ['Python tutorial', 'LangChain guide', 'machine learning']
[Usage Tracker] User: user_001, Tier: enterprise

🔄 Node: track_usage.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 

🔄 Node: model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

김철수님, 당신의 최근 검색 기록은 다음과 같습니다:



In [7]:
# Free 한도 초과: track_usage 가 jump_to="end" 로 모델 호출 없이 종료
print("=== Free tier: limit exceeded (jump_to end) ===")
result = workflow.invoke(
    inputs={"messages": [HumanMessage(content="추가 검색 요청")]},
    context=UserContext(
        user_id="user_limit",
        user_name="한도초과",
        user_tier="free",
        language="ko",
    ),
)
print(result["messages"][-1].content)

=== Free tier: limit exceeded (jump_to end) ===
[Usage Tracker] User: user_limit, Tier: free
[Usage Tracker] Free tier limit reached!

🔄 Node: track_usage.before_model 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
jump_to:
end
================================== Ai Message ==================================

무료 플랜 모델 호출 한도(10회)에 도달했습니다. (현재 사용량: 10회)
무료 플랜 모델 호출 한도(10회)에 도달했습니다. (현재 사용량: 10회)
